# Stale State Recovery

**Last updated:** 2026-04-15 — PT

**Track:** Learning | **Sub-Ability:** Procedural learning

Can the model learn — from a small number of worked examples — a world model in which references to state become *stale* after mutation, and then apply that model to debug and repair a new operation sequence?

The failure mode this targets is a real one seen in agent traces: `Ref e803 not found` after a page change, `File has been modified since read` after an external edit. Agents re-use stale refs without re-acquiring state.

**Operations:**
- `snapshot()` — captures the current world state and returns a fresh ref token
- `act(ref.item_X)` — acts using a ref (FAILS if the ref is stale)
- `mutate()` — changes world state (invalidates ALL prior refs)
- Some variants: named operations (e.g. `write_file()`, `commit()`) secretly mutate as a side-effect

**Task:** Given a sequence with a stale-ref bug, output the corrected sequence (with a re-snapshot inserted at the right place).

**Difficulty grid:** complexity (`single_mutation`, `multiple_mutations`, `implicit_mutation`) × evidence (`few`=3ex, `mid`=5ex, `many`=6ex) × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def _normalize_line(line: str) -> str:
    """Normalize a single op line for comparison.

    Strips the step number prefix (e.g. '5. '), any trailing comment after
    '#', collapses whitespace, and lowercases. This lets us compare semantic
    op content even if a model paraphrases comments.
    """
    s = line.strip()
    # Strip leading step number like '5.' / '5)' / '5:'
    s = re.sub(r'^\s*\d+\s*[\.\):]\s*', '', s)
    # Strip trailing comment
    if '#' in s:
        s = s.split('#', 1)[0]
    # Collapse whitespace, remove backticks/markdown noise
    s = re.sub(r'[`*]', '', s)
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s


def extract_sequence(response: str) -> str:
    """Extract a corrected op sequence from the model response.

    Heuristic: find the longest contiguous block of lines that look like
    numbered op lines (starting with 'N.' or 'N)' prefix). Return those
    normalized and rejoined with newlines.
    """
    text = strip_thinking(response)
    # Strip code fences if present
    text = re.sub(r'```[a-zA-Z]*\n', '', text)
    text = text.replace('```', '')
    lines = text.split('\n')
    best_block = []
    cur_block = []
    for line in lines:
        stripped = line.strip()
        if re.match(r'^\d+\s*[\.\):]', stripped):
            cur_block.append(stripped)
        else:
            if len(cur_block) > len(best_block):
                best_block = cur_block
            cur_block = []
    if len(cur_block) > len(best_block):
        best_block = cur_block
    # Normalize each line
    norm = [_normalize_line(l) for l in best_block if _normalize_line(l)]
    return '\n'.join(norm)


def normalize_expected(expected: str) -> str:
    """Normalize the gold sequence the same way as extracted answer."""
    lines = [l for l in expected.split('\n') if l.strip()]
    norm = [_normalize_line(l) for l in lines if _normalize_line(l)]
    return '\n'.join(norm)

In [ ]:
# === Constants for results analysis ===
COMPLEXITY_LEVELS = ['single_mutation', 'multiple_mutations', 'implicit_mutation']
EVIDENCE_LEVELS = ['few', 'mid', 'many']
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/stale_state_recovery_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/stale_state_recovery_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id', 'difficulty_label', 'seed', 'item_desc']].to_string(index=False))

In [ ]:
PRE_P = """You are analyzing an operation sequence in an abstract world. Some worked examples below demonstrate how operations work; study them to learn the rules. Then fix the buggy sequence.

--- WORKED EXAMPLES ---
{material}
--- END EXAMPLES ---

Now here is a new operation sequence that contains a stale-reference bug (one or more `act(ref.*)` calls use a ref that has been invalidated by an earlier operation):

{test_input}

Your task:
1. Identify the step number where the stale-ref bug occurs.
2. Output the CORRECTED sequence with the necessary re-snapshot inserted so that every `act(ref.*)` uses a valid ref. Re-number the steps.

Use fresh ref names (s2, s3, ...) when you insert a new snapshot. Keep all other operations and their order.

Give your final answer as a numbered list of ops, one per line, starting with `1.`. Put the corrected sequence on its own lines with no surrounding prose."""

STUDY_P = """You are learning a small world model from worked examples.

--- WORKED EXAMPLES ---
{material}
--- END EXAMPLES ---

Your task:
1. State the rule(s) for when a ref becomes stale. Be precise about which operations invalidate refs.
2. State the rule for recovery (what must be inserted and where).
3. If any operations in the examples are named things other than `snapshot`, `act`, or `mutate`, list which ones invalidate refs and which ones are safe. Cite the evidence in the examples.
4. Walk through one example and annotate which steps would fail if a needed re-snapshot were removed.
5. Write a short checklist you will follow when debugging a new buggy sequence.

Show all your work."""

POST_P = """You previously studied a small world model and wrote these notes:

--- YOUR STUDY NOTES ---
{notes}
--- END NOTES ---

Here are the worked examples again for reference:
--- WORKED EXAMPLES ---
{material}
--- END EXAMPLES ---

Now here is a new operation sequence that contains a stale-reference bug (one or more `act(ref.*)` calls use a ref that has been invalidated by an earlier operation):

{test_input}

Your task:
1. Identify the step number where the stale-ref bug occurs.
2. Output the CORRECTED sequence with the necessary re-snapshot inserted so that every `act(ref.*)` uses a valid ref. Re-number the steps.

Use fresh ref names (s2, s3, ...) when you insert a new snapshot. Keep all other operations and their order.

Give your final answer as a numbered list of ops, one per line, starting with `1.`. Put the corrected sequence on its own lines with no surrounding prose."""

## Evaluation

In [ ]:
all_results = []

@kbench.task(name='stale_state_recovery',
             description='Pre/post learning a stale-ref world model; identify invalidation point and output a corrected op sequence')
def stale_state_recovery(llm, material: str, test_input: str, expected: str,
                         complexity: str, evidence: str, difficulty_label: str,
                         seed: int, item_desc: str, stale_step: int) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{complexity}_{evidence}_{seed}'
    expected_norm = normalize_expected(expected)

    # PRE: solve with only the worked examples
    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
    pre_time = time.time() - t0
    pre_extracted = extract_sequence(pre_raw)
    pre_correct = pre_extracted == expected_norm

    # STUDY: internalize the rules
    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    # POST: solve with the study notes in hand
    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
    post_time = time.time() - t0
    post_extracted = extract_sequence(post_raw)
    post_correct = post_extracted == expected_norm

    all_results.append({
        'timestamp': ts, 'model': model_name, 'task_id': tid,
        'complexity': complexity, 'evidence': evidence, 'difficulty_label': difficulty_label,
        'seed': int(seed), 'item_desc': item_desc, 'stale_step': int(stale_step),
        'material': material, 'test_input': test_input, 'expected': expected,
        'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
        'study_notes': notes,
        'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
        'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time,
    })

    b, l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{difficulty_label:26s}] seed={seed} stale@{stale_step}  pre={b}  post={l}')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

try:
    runs = stale_state_recovery.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                          n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

# Per difficulty label
print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:26s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per complexity
print()
print('By Complexity:')
print('-' * 60)
for comp in COMPLEXITY_LEVELS:
    sub = results[results['complexity'] == comp]
    if len(sub) == 0: continue
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {comp:22s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per evidence level
print()
print('By Evidence:')
print('-' * 60)
for ev in EVIDENCE_LEVELS:
    sub = results[results['evidence'] == ev]
    if len(sub) == 0: continue
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {ev:8s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per item
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    delta = '  HELPED' if not row['pre_correct'] and row['post_correct'] else \
            '  HURT' if row['pre_correct'] and not row['post_correct'] else ''
    print(f'  [{row["difficulty_label"]:26s}] seed={row["seed"]} stale@{row["stale_step"]:<2d}  pre={b}  post={l}{delta}')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} stale@{row["stale_step"]} ---')
    print(f'Pre correct: {b}  Post correct: {l}')
    print(f'Notes (first 400 chars): {row["study_notes"][:400]}')
    print('...' if len(str(row['study_notes'])) > 400 else '')
    print(f'Post extracted (first 300 chars): {row["post_extracted"][:300]}')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Stale State Recovery: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('stale_state_recovery_results.png', dpi=150, bbox_inches='tight')
plt.show()